In [ ]:
# NCrystal notebook settings
# title: Building a McStas instrument with NCrystal in Python
# menutitle: McStas instrument in Python
# shortkey: mcstasantlrapi
# section: misc
# requires: plot, mccodeantlr, scipp
# test-parameters: NParticles = 100000

## Introduction [intro]

In this example we simulate a simple transmission beamline with an NCrystal sample in McStas, using the Python package [mccode-antlr](https://github.com/mccode-dev/mccode-antlr), which can be installed with pip. Unlike the notebook "NCrystal materials in McStas with mccode-antlr", where the instrument is written as McStas instrument code, here we build the instrument step by step with Python objects. This makes it easy to modify the instrument from Python, for instance to add or move components. We then compile and run the simulation, and get its results as [scipp](https://scipp.github.io) data arrays (with units and uncertainties), which we plot with [plopp](https://scipp.github.io/plopp).

Compiling the instrument needs a C compiler. One is available on Google Colab and on most Linux systems, and on macOS it comes with the Xcode command line tools.

### Additional modules: [modules]

In [ ]:
import pathlib
import shutil
import numpy as np
import scipp as sc
import plopp as pp
from mccode_antlr import Flavor
from mccode_antlr.assembler import Assembler
from mccode_antlr.run import McStas
#mccode-antlr logs many details (e.g. each component it reads), so we only
#show its warnings and errors:
import sys
from loguru import logger
logger.remove()
logger.add(sys.stderr, level='WARNING')

## The instrument [instrument]

We use a few variables for the dimensions of the beamline and for the simulation:

In [ ]:
beam_width = 0.01  # m
beam_height = 0.01  # m
sample_thickness = 0.01  # m
nbins = 700
min_lambda = 1.0  # Å
max_lambda = 8.0  # Å
sample_cfgstr = 'C_sg194_pyrolytic_graphite.ncmat;temp=20C'
NParticles = int(1e6)  # number of neutrons to simulate

The instrument is built with an `Assembler`. The NCrystal cfg-string of the sample is an instrument parameter, so it can be changed without compiling the instrument again:

In [ ]:
a = Assembler('NCrystal_transm_example', flavor=Flavor.MCSTAS)
a.parameter('string sample_cfg = "Al_sg225.ncmat;temp=300K"')

Then we add the components, each with a name, the name of the McStas component (downloaded by mccode-antlr when the instrument is compiled), its position, and its parameters. Positions are given relative to a previous component. First a source of neutrons with wavelengths between `min_lambda` and `max_lambda`, focused on a beam of the given size:

In [ ]:
origin = a.component('origin', 'Progress_bar', at=(0, 0, 0))
source = a.component('source', 'Source_gen', at=([0, 0, 0], origin),
                     parameters=dict(Lmin=min_lambda, Lmax=max_lambda,
                                     xwidth=beam_width, yheight=beam_height,
                                     focus_xw=beam_width, focus_yh=beam_height,
                                     dist=100))

The wavelength monitors before and after the sample are identical, so we add them with a small function. Values for string parameters of components must be given with quotes (as in McStas instrument code), like the file name here:

In [ ]:
def add_monitor(name, at):
    return a.component(name, 'L_monitor', at=at,
                       parameters=dict(xmin=-beam_width/2, xmax=beam_width/2,
                                       ymin=-beam_height/2, ymax=beam_height/2,
                                       nL=nbins, Lmin=min_lambda, Lmax=max_lambda,
                                       filename=f'"{name}.dat"',
                                       restore_neutron=1))

monitor1 = add_monitor('monitor1', at=([0, 0, 0.5], source))

The sample is a slab of the material given by the instrument parameter `sample_cfg`. With `split=10`, each neutron reaching the sample is traced through it 10 times, which improves the statistics after the sample:

In [ ]:
sample = a.component('sample', 'NCrystal_sample', at=([0, 0, 0.5], monitor1),
                     parameters=dict(xwidth=10.0, yheight=10.0,
                                     zdepth=sample_thickness, cfg='sample_cfg'),
                     split=10)
monitor2 = add_monitor('monitor2', at=([0, 0, 0.5], sample))

The instrument is now complete. Shown in the notebook, it gives an overview of its parameters and components:

In [ ]:
instr = a.instrument
instr

## Running the simulation [run]

We compile the instrument in the directory `build`, and run it with the NCrystal cfg-string of our sample. The results are written to a new directory (so we remove it first, in case the notebook is run again):

In [ ]:
sim = McStas(instr).compile('./build')

outdir = pathlib.Path('./rundir')
shutil.rmtree(outdir, ignore_errors=True)
result = sim.run({'sample_cfg': sample_cfgstr}, ncount=NParticles,
                 output_dir=outdir)
print(result.stdout)

## Results [results]

The data of the monitors are available by their names. The intensities are scipp data arrays, with units, uncertainties, and the wavelength of each bin:

In [ ]:
intensity1 = result.dats['monitor1']['I']
intensity2 = result.dats['monitor2']['I']
intensity1

In [ ]:
pp.plot({'Before the sample (monitor1)': intensity1,
         'After the sample (monitor2)': intensity2})

The transmission through the sample is the ratio of the two intensities (scipp takes care of the units and the uncertainties). We compare it with the theoretical transmission of neutrons not interacting in the sample, computed with NCrystal:

In [ ]:
transmission = intensity2 / intensity1

mat = NC.load(sample_cfgstr)
wl = np.linspace(min_lambda, max_lambda, 2000)
xsect = mat.scatter.xsect(wl=wl) + mat.absorption.xsect(wl=wl)
sample_thickness_cm = sample_thickness * 100
theory = sc.DataArray(
    sc.array(dims=['Wavelength'],
             values=np.exp(-mat.info.getNumberDensity()*xsect*sample_thickness_cm)),
    coords={'Wavelength': sc.array(dims=['Wavelength'], values=wl, unit='Å')})

pp.plot({'McStas simulation': transmission, 'Theory (uncollided)': theory},
        linestyle={'McStas simulation': 'none', 'Theory (uncollided)': 'solid'},
        marker={'McStas simulation': '.', 'Theory (uncollided)': None},
        title='Transmission through the sample')

## Exercise [exercise]

- Change the sample: try other materials, temperatures or densities, by changing `sample_cfgstr` (no new compilation needed).
- Modify the instrument, for instance by moving the second monitor closer to the sample to study the effect of in-scattering, or by adding a component, and compile and run it again.

In [ ]:
## Your code